In [1]:
import geopandas as gpd
import pandas as pd
import os
import fiona

# Define the directory containing your GDB files
source_dir = r'C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\XYplots'

# Loop through the directory to find GDB folders
for item in os.listdir(source_dir):
    if item.endswith('.gdb'):
        gdb_path = os.path.join(source_dir, item)
        
        # Define output path (same name as GDB, but .xlsx)
        output_file_name = item.replace('.gdb', '.xlsx')
        output_path = os.path.join(source_dir, output_file_name)
        
        try:
            # List all layers inside the GDB
            layers = fiona.listlayers(gdb_path)
            print(f"Processing {item}: Found {len(layers)} layers.")

            # Use ExcelWriter to handle multiple sheets
            with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
                for layer_name in layers:
                    # Read the specific layer
                    gdf = gpd.read_file(gdb_path, layer=layer_name)
                    
                    # Skip empty layers
                    if gdf.empty or gdf.geometry.isnull().all():
                        print(f"  Skipping empty layer: {layer_name}")
                        continue
                    
                    # Calculate Centroids (X and Y coordinates)
                    gdf['X'] = gdf.geometry.centroid.x
                    gdf['Y'] = gdf.geometry.centroid.y
                    
                    # Drop geometry column for Excel compatibility
                    df_layer = pd.DataFrame(gdf.drop(columns='geometry'))
                    
                    # Write to a specific sheet named after the layer
                    # Excel sheet names have a 31-character limit
                    sheet_name = layer_name[:31] 
                    df_layer.to_excel(writer, sheet_name=sheet_name, index=False)
                    print(f"  Layer '{layer_name}' written to sheet.")

            print(f"--- Success! All layers for {item} saved to separate sheets in {output_file_name}")
            
        except Exception as e:
            print(f"Error processing {item}: {e}")

print("\nMulti-sheet extraction complete.")

Processing LODS24.gdb: Found 2 layers.
  Layer 'P24116' written to sheet.
  Layer 'N24116' written to sheet.
--- Success! All layers for LODS24.gdb saved to separate sheets in LODS24.xlsx
Processing LODS25.gdb: Found 9 layers.
  Layer 'P25111' written to sheet.
  Layer 'P25112' written to sheet.
  Layer 'P25113' written to sheet.
  Layer 'P25114' written to sheet.
  Layer 'P25117' written to sheet.
  Layer 'P25110' written to sheet.
  Layer 'P25104' written to sheet.
  Layer 'P25105' written to sheet.
  Layer 'P25106' written to sheet.
--- Success! All layers for LODS25.gdb saved to separate sheets in LODS25.xlsx
Processing MRF2021.gdb: Found 3 layers.
  Layer 'pLOT21MRC' written to sheet.
  Layer 'PLOT2115' written to sheet.
  Layer 'PLOT2116' written to sheet.
--- Success! All layers for MRF2021.gdb saved to separate sheets in MRF2021.xlsx
Processing MRF2022.gdb: Found 3 layers.
  Layer 'Plots2022MRC2217' written to sheet.
  Layer 'Plots2022MRC2218' written to sheet.
  Layer 'Plots20

In [5]:
import pandas as pd
import os
import numpy as np
import re

# Define file paths
master_file_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_kg_ha_updated.xlsx"
xy_dir = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\XYplots"
output_path = r"C:\Users\bazrafka\Desktop\counting\DiscussionPaperData\Python\Ali_Data_with_XY_Final_Fixed.xlsx"

# Load master dataframe
master_df = pd.read_excel(master_file_path)

# Ensure PlotID is numeric
master_df['PlotID'] = pd.to_numeric(master_df['PlotID'], errors='coerce').fillna(-1).astype(int)

# --- IMPROVED MATCH KEY LOGIC ---
def extract_numeric_id(text):
    """Extracts only the digits from a string (e.g., SEVREC2505 -> 2505)"""
    digits = re.findall(r'\d+', str(text))
    return "".join(digits) if digits else ""

# Apply numeric extraction to master Experiment Name
master_df['Match_ID'] = master_df['Experiment Name'].apply(extract_numeric_id)

# List to collect all XY data
xy_data_list = []

for file_name in os.listdir(xy_dir):
    if file_name.endswith('.xlsx'):
        file_path = os.path.join(xy_dir, file_name)
        xl = pd.ExcelFile(file_path)
        
        for sheet in xl.sheet_names:
            df_xy = pd.read_excel(file_path, sheet_name=sheet)
            
            if 'PlotID' in df_xy.columns and 'X' in df_xy.columns and 'Y' in df_xy.columns:
                # Ensure PlotID is numeric
                df_xy['PlotID'] = pd.to_numeric(df_xy['PlotID'], errors='coerce').fillna(-2).astype(int)
                
                # Extract numeric ID from sheet name (e.g., SEVREC2505 -> 2505)
                df_xy['Match_ID'] = extract_numeric_id(sheet)
                
                xy_data_list.append(df_xy[['PlotID', 'Match_ID', 'X', 'Y']])

# Combine and remove duplicates
full_xy_df = pd.concat(xy_data_list, ignore_index=True)
full_xy_df = full_xy_df.drop_duplicates(subset=['PlotID', 'Match_ID'])

# Perform the final merge
final_df = pd.merge(
    master_df,
    full_xy_df,
    on=['PlotID', 'Match_ID'],
    how='left'
)

# Cleanup and save
final_df = final_df.drop(columns=['Match_ID'])
final_df.to_excel(output_path, index=False)

print(f"Join complete using numeric extraction logic.")
print(f"Total plots in master: {len(master_df)}")
print(f"Plots with assigned X/Y: {final_df['X'].notna().sum()}")

Join complete using numeric extraction logic.
Total plots in master: 4512
Plots with assigned X/Y: 4512
